#PreProcess

In [9]:
import torch
import pandas as pd
import numpy as np

def load_data(file_path, chunk_size=10000):
    """Loads data from a large CSV file in chunks."""
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        yield chunk

def preprocess_chunk(chunk):
    """Processes a chunk of data: replace NaNs, create mask, calculate means, create conditional vector c."""
    # 1. Replace NaNs with 0
    chunk_filled = chunk.fillna(0)

    # 2. Create a binary mask
    mask = (~chunk.isna()).astype(float)

    # 3. Calculate per-sample column means for observed values
    # Use .values to avoid potential issues with index alignment during division
    observed_values = chunk.values * mask.values
    sum_observed = np.nansum(observed_values, axis=1)
    num_observed = np.sum(mask.values, axis=1)
    # Avoid division by zero
    per_sample_means = np.divide(sum_observed, num_observed, out=np.zeros_like(sum_observed), where=num_observed != 0)
    per_sample_means = per_sample_means[:, np.newaxis] # Reshape to (n_samples, 1)

    # 4. Create conditional vector c by combining mask and means
    # Ensure shapes are compatible for concatenation
    c = np.concatenate([mask.values, per_sample_means], axis=1)

    return chunk_filled, mask, c

#DataSet

In [10]:
class LikertDataset(torch.utils.data.IterableDataset):
    """Custom PyTorch IterableDataset for Likert scale data."""
    def __init__(self, file_path, chunk_size=10000):
        super().__init__()
        self.file_path = file_path
        self.chunk_size = chunk_size

    def __iter__(self):
        # Get worker info if using multiple workers in DataLoader
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:  # Single-process data loading
            start = 0
            end = float('inf')
        else:  # Multiple workers
            per_worker = int(np.ceil((self._get_total_rows() or 1e6) / float(worker_info.num_workers)))
            worker_id = worker_info.id
            start = worker_id * per_worker
            end = min(start + per_worker, (self._get_total_rows() or 1e6))

        # Iterate through data chunks
        for i, chunk in enumerate(load_data(self.file_path, self.chunk_size)):
             # Simple chunk distribution based on index (can be improved)
            if i * self.chunk_size >= end:
                break
            if (i + 1) * self.chunk_size <= start and i * self.chunk_size < start:
                continue

            chunk_filled, mask, c = preprocess_chunk(chunk)

            # Yield each sample from the processed chunk
            for j in range(len(chunk_filled)):
                 yield (torch.FloatTensor(chunk_filled.iloc[j].values),
                       torch.FloatTensor(mask.iloc[j].values),
                       torch.FloatTensor(c[j]))

    # Helper to get total rows (optional, can be slow for very large files)
    def _get_total_rows(self):
        try:
            return sum(1 for row in open(self.file_path, 'r')) - 1 # Subtract header
        except:
            return None # Return None if unable to get row count


#OrdinalEmbedding

In [11]:
import torch
import torch.nn as nn
import torch.distributions as distributions

class OrdinalEmbedding(nn.Module):
    """Maps Likert scale values (1-5) to dense embedding vectors."""
    def __init__(self, embedding_dim):
        super().__init__()
        # Embedding for values 1, 2, 3, 4, 5. Index 0 can be padding or unused.
        self.embedding = nn.Embedding(6, embedding_dim) # Size 6 for indices 0-5

    def forward(self, x):
        # Assuming input x is a tensor of integers representing Likert values (1-5)
        # If input is 0 (for masked values), the embedding will be for index 0.
        return self.embedding(x.long())

class Encoder(nn.Module):
    """Encodes embedded input items and conditional vector c into latent distribution parameters."""
    def __init__(self, input_dim, c_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim + c_dim, 256) # Example hidden layer size
        self.fc2 = nn.Linear(256, 128) # Example hidden layer size
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.relu = nn.ReLU()

    def forward(self, embedded_x, c):
        # Concatenate embedded input and conditional vector
        h = torch.cat([embedded_x, c], dim=-1) # Concatenate along the last dimension

        h = self.relu(self.fc1(h))
        h = self.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    """Decodes latent vector z and conditional vector c into 5-class softmax logits for each item."""
    def __init__(self, latent_dim, c_dim, output_dim, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim + c_dim, 128) # Example hidden layer size
        self.fc2 = nn.Linear(128, 256) # Example hidden layer size
        # Output layer: produces logits for 5 classes per item
        # output_dim is the number of items (e.g., 50)
        self.fc_out = nn.Linear(256, output_dim * num_classes)
        self.relu = nn.ReLU()
        self.output_dim = output_dim
        self.num_classes = num_classes

    def forward(self, z, c):
        # Concatenate latent vector and conditional vector
        h = torch.cat([z, c], dim=-1) # Concatenate along the last dimension

        h = self.relu(self.fc1(h))
        h = self.relu(self.fc2(h))
        logits = self.fc_out(h)
        # Reshape logits to (batch_size, output_dim, num_classes)
        logits = logits.view(-1, self.output_dim, self.num_classes)
        return logits



CVAE

In [12]:
class CVAE(nn.Module):
    """Conditional Variational Autoencoder for Likert scale data."""
    def __init__(self, num_items, embedding_dim, c_dim, latent_dim):
        super().__init__()
        self.ordinal_embedding = OrdinalEmbedding(embedding_dim)
        # Input dimension for encoder is num_items * embedding_dim
        self.encoder = Encoder(num_items * embedding_dim, c_dim, latent_dim)
        self.decoder = Decoder(latent_dim, c_dim, num_items, num_classes=5)
        self.num_items = num_items
        self.latent_dim = latent_dim

    def forward(self, x, c):
        # x: input Likert values (batch_size, num_items)
        # c: conditional vector (batch_size, c_dim)

        # Embed each item in the input
        # embedded_x shape: (batch_size, num_items, embedding_dim)
        embedded_x = self.ordinal_embedding(x)

        # Flatten the embedded_x for the encoder
        # flattened_embedded_x shape: (batch_size, num_items * embedding_dim)
        flattened_embedded_x = embedded_x.view(embedded_x.size(0), -1)

        # Get latent distribution parameters from encoder
        mu, logvar = self.encoder(flattened_embedded_x, c)

        # Sample from the latent distribution using reparameterization trick
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std

        # Decode the latent vector and conditional vector
        # logits shape: (batch_size, num_items, num_classes)
        logits = self.decoder(z, c)

        return logits, mu, logvar

#LossFunction

In [13]:
import torch.nn.functional as F

def ordinal_cross_entropy_loss(logits, targets, mask):
    """
    Calculates the cross-entropy loss only for the observed values (1-5).

    Args:
        logits (torch.Tensor): Predicted logits (batch_size, num_items, num_classes).
        targets (torch.Tensor): True Likert values (batch_size, num_items).
        mask (torch.Tensor): Binary mask (batch_size, num_items), 1 for observed, 0 for missing.

    Returns:
        torch.Tensor: Scaled mean cross-entropy loss over observed values.
    """
    # Flatten tensors
    logits_flat = logits.view(-1, logits.size(-1)) # (batch_size * num_items, num_classes)
    targets_flat = targets.view(-1) # (batch_size * num_items)
    mask_flat = mask.view(-1) # (batch_size * num_items)

    # Create a combined mask: originally observed AND target value is within 1-5 range
    valid_observed_mask = (mask_flat == 1) & (targets_flat >= 1) & (targets_flat <= 5)

    # Apply combined mask: only consider valid observed values
    observed_logits = logits_flat[valid_observed_mask]
    observed_targets = targets_flat[valid_observed_mask]

    num_observed = observed_targets.size(0)

    if num_observed == 0:
        return torch.tensor(0.0, device=logits.device)

    # Ensure observed targets are long type and adjust from 1-5 to 0-4 for cross_entropy index
    # Now we are sure observed_targets are in [1, 5] range
    observed_targets = observed_targets.long() - 1

    # Calculate cross-entropy loss for observed items
    reconstruction_loss = F.cross_entropy(observed_logits, observed_targets, reduction='mean')

    return reconstruction_loss

def kl_divergence_loss(mu, logvar, free_bits=0.1):
    """
    Calculates the KL divergence loss with free bits.

    Args:
        mu (torch.Tensor): Mean of the latent distribution.
        logvar (torch.Tensor): Log variance of the latent distribution.
        free_bits (float): The number of free bits.

    Returns:
        torch.Tensor: KL divergence loss.
    """
    # KL divergence between N(mu, var) and N(0, 1)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Apply free bits: the KL loss is not penalized if it's below free_bits
    kl_loss = torch.max(kl_loss, torch.tensor(free_bits * mu.size(0), device=mu.device)) # Scale free_bits by batch size

    return kl_loss

def combined_loss(reconstruction_loss, kl_loss, beta):
    """
    Combines reconstruction loss and KL divergence loss with beta weighting.

    Args:
        reconstruction_loss (torch.Tensor): The reconstruction loss.
        kl_loss (torch.Tensor): The KL divergence loss.
        beta (float): The weight for the KL divergence loss.

    Returns:
        torch.Tensor: The total loss.
    """
    return reconstruction_loss + beta * kl_loss

#CalculateMetrics

In [14]:
def calculate_metrics(logits, targets, mask):
    """
    Calculates Accuracy, MAE, and RMSE for observed values.

    Args:
        logits (torch.Tensor): Predicted logits (batch_size, num_items, num_classes).
        targets (torch.Tensor): True Likert values (batch_size, num_items).
        mask (torch.Tensor): Binary mask (batch_size, num_items).

    Returns:
        tuple: (accuracy, mae, rmse)
    """
    # Get predicted Likert values (1-5) from logits
    # Find the class with the highest logit for each item
    predicted_classes = torch.argmax(logits, dim=-1) # (batch_size, num_items)
    predicted_values = predicted_classes + 1 # Adjust predicted classes from 0-4 to 1-5

    # Flatten tensors and apply mask
    predicted_flat = predicted_values.view(-1)
    targets_flat = targets.view(-1)
    mask_flat = mask.view(-1)

    # Apply mask to keep only observed values
    observed_predicted = predicted_flat[mask_flat == 1]
    observed_targets = targets_flat[mask_flat == 1]

    num_observed = observed_targets.size(0)

    if num_observed == 0:
        return 0.0, 0.0, 0.0 # Return 0 if no observed values

    # Accuracy: percentage of correctly predicted values among observed
    correct_predictions = (observed_predicted == observed_targets).sum().item()
    accuracy = correct_predictions / num_observed

    # MAE: Mean Absolute Error among observed
    mae = torch.abs(observed_predicted.float() - observed_targets.float()).mean().item()

    # RMSE: Root Mean Squared Error among observed
    rmse = torch.sqrt(torch.mean((observed_predicted.float() - observed_targets.float())**2)).item()

    return accuracy, mae, rmse

def apply_artificial_masking(data, mask, masking_prob):
    """
    Applies artificial masking to the data for evaluation purposes.

    Args:
        data (torch.Tensor): Input data (batch_size, num_items).
        mask (torch.Tensor): Original mask (batch_size, num_items).
        masking_prob (float): Probability of masking an originally observed value.

    Returns:
        tuple: (artificially_masked_data, artificial_mask)
    """
    # Create a new mask starting with the original mask
    artificial_mask = mask.clone()

    # Identify originally observed values
    observed_indices = (mask == 1)

    # Create a random mask for observed values based on masking_prob
    random_mask = torch.rand_like(data, dtype=torch.float) < masking_prob

    # Combine the random mask with the observed indices to only mask originally observed values
    mask_to_apply = observed_indices * random_mask

    # Apply artificial masking: set masked values to 0 in the data
    artificially_masked_data = data.clone()
    artificially_masked_data[mask_to_apply] = 0

    # Update the artificial mask: set masked positions to 0
    artificial_mask[mask_to_apply] = 0

    return artificially_masked_data, artificial_mask

# Placeholder for training loop, warm-up, and early stopping which will be implemented next
# Need to define model, optimizer, dataloaders, and hyperparameters first.
# This part of the code block only defines the necessary functions.

**Reasoning**:
Implement the training loop including beta warm-up, validation, and early stopping, utilizing the previously defined model and loss/evaluation functions. This completes the final part of the subtask related to training and evaluation.



#HyperParameters

In [15]:
from google.colab import drive
drive.mount('/content/drive')

# Define hyperparameters (example values - should be tuned)
FILE_PATH = "/content/drive/MyDrive/Projects/CVAE_MI/big5_ans_only.csv"
CHUNK_SIZE = 10000
BATCH_SIZE = 128
EMBEDDING_DIM = 32
LATENT_DIM = 64
NUM_ITEMS = 50 # Assuming 50 items in the Likert scale data
C_DIM = NUM_ITEMS + 1 # Mask (num_items) + Mean (1)
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100 # Set a reasonable number of epochs
BETA_START = 0.0
BETA_END = 1.0
BETA_WARMUP_EPOCHS = 10
FREE_BITS = 0.01
EARLY_STOPPING_PATIENCE = 20 # Number of epochs to wait for improvement
MASKING_PROB_EVAL = 0.1 # Probability for artificial masking during evaluation


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Loop

In [ ]:
# import torch_xla.core.xla_model as xm # Remove TPU import
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
# from tqdm.auto import tqdm # Remove tqdm

# Set device - prioritize CUDA, then default to CPU
# Removed TPU specific device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Dataset and DataLoader
# For large datasets, splitting into train/val might require iterating or saving processed chunks
# For simplicity in this example, let's assume we can iterate through the dataset and split manually or use separate files
# A more robust solution for large data would involve pre-processing and saving to a format like HDF5 or Parquet, then using a standard Map-style Dataset
# Here, we'll use the IterableDataset for demonstration, but a proper train/val split with IterableDataset can be tricky.
# Let's simulate a split by iterating a few chunks for validation.

# Initialize the full dataset
full_dataset = LikertDataset(FILE_PATH, chunk_size=CHUNK_SIZE)

# To create train/val split with IterableDataset, one approach is to create two instances
# pointing to potentially different files, or implementing split logic within __iter__
# Given the constraints, let's just use the same dataset for train/val for demonstration,
# which is not ideal for proper evaluation but serves to show the training loop structure.
# In a real scenario, you would need separate train/val data sources.
train_dataset = full_dataset # Replace with actual train dataset
val_dataset = full_dataset   # Replace with actual val dataset

# With IterableDataset, DataLoader does not have a __len__
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


# Initialize model, optimizer
model = CVAE(NUM_ITEMS, EMBEDDING_DIM, C_DIM, LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training Loop
best_val_loss = float('inf')
epochs_without_improvement = 0
global_step = 0

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    train_reconstruction_loss = 0.0
    train_kl_loss = 0.0
    num_train_batches = 0

    # Calculate current beta for KL warm-up
    beta = BETA_START + (BETA_END - BETA_START) * min(1.0, epoch / BETA_WARMUP_EPOCHS)

    # Wrap train_loader with tqdm for progress bar
    # train_loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Training") # Removed tqdm

    for i, (x, mask, c) in enumerate(train_loader):
        x, mask, c = x.to(device), mask.to(device), c.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits, mu, logvar = model(x, c)

        # Calculate losses
        reconstruction_loss = ordinal_cross_entropy_loss(logits, x, mask)
        kl_loss = kl_divergence_loss(mu, logvar, free_bits=FREE_BITS)
        loss = combined_loss(reconstruction_loss, kl_loss, beta)

        # Backward pass and optimize
        loss.backward()
        # Use xm.optimizer_step(optimizer) for TPU
        # Removed TPU specific optimizer step
        # if device.type == 'xla':
        #     xm.optimizer_step(optimizer)
        # else:
        optimizer.step()

        train_loss += loss.item()
        train_reconstruction_loss += reconstruction_loss.item()
        train_kl_loss += kl_loss.item() * beta # Store the beta-weighted KL for reporting

        global_step += 1
        num_train_batches += 1

        # Print batch progress periodically
        if (i + 1) % 1000 == 0: # Print every 1000 batches
             print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}], Batch [{i+1}], Loss: {loss.item():.4f}, Rec Loss: {reconstruction_loss.item():.4f}, KL Loss (weighted): {kl_loss.item() * beta:.4f}, Beta: {beta:.4f}")


    # Average training loss for the epoch - Divide by number of batches processed
    avg_train_loss = train_loss / num_train_batches if num_train_batches > 0 else 0.0
    avg_train_reconstruction_loss = train_reconstruction_loss / num_train_batches if num_train_batches > 0 else 0.0
    avg_train_kl_loss = train_kl_loss / num_train_batches if num_train_batches > 0 else 0.0


    # Validation Step
    model.eval()
    val_loss = 0.0
    val_reconstruction_loss = 0.0
    val_kl_loss = 0.0
    val_accuracy = 0.0
    val_mae = 0.0
    val_rmse = 0.0
    val_accuracy_masked = 0.0
    val_mae_masked = 0.0
    val_rmse_masked = 0.0
    num_val_batches = 0

    # Wrap val_loader with tqdm for progress bar
    # val_loop = tqdm(val_loader, leave=False, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Validation") # Removed tqdm

    with torch.no_grad():
        for i, (x, mask, c) in enumerate(val_loader):
            x, mask, c = x.to(device), mask.to(device), c.to(device)

            # Calculate validation loss on original data
            logits, mu, logvar = model(x, c)
            reconstruction_loss = ordinal_cross_entropy_loss(logits, x, mask)
            kl_loss = kl_divergence_loss(mu, logvar, free_bits=FREE_BITS)
            loss = combined_loss(reconstruction_loss, kl_loss, beta) # Use current beta for consistency

            val_loss += loss.item()
            val_reconstruction_loss += reconstruction_loss.item()
            val_kl_loss += kl_loss.item() * beta

            # Calculate metrics on original observed data
            acc, mae, rmse = calculate_metrics(logits, x, mask)
            val_accuracy += acc
            val_mae += mae
            val_rmse += rmse

            # Apply artificial masking and calculate metrics on artificially masked data
            # Note: c should ideally be re-calculated based on artificially masked data + original means
            # For simplicity here, we'll use the original c, which is a limitation for evaluation
            # on artificially masked *input*. The metrics are calculated on the reconstruction
            # of the artificially masked values.
            artificially_masked_x, artificial_mask = apply_artificial_masking(x, mask, MASKING_PROB_EVAL)
            # Note: The CVAE model in this implementation takes the original x (possibly with NaNs filled)
            # as input to the encoder, not the artificially masked_x.
            # To evaluate the model's ability to predict artificially masked values,
            # we use the logits produced by the forward pass with the *original* input x,
            # but evaluate the metrics specifically on the positions that were *artificially masked*.
            # The 'artificial_mask' here helps us identify which positions were masked.

            # Identify positions that were originally observed AND are now artificially masked
            # The artificial_mask is 0 where values were masked or artificially masked
            # We want to evaluate performance on positions where original mask was 1 and artificial_mask is 0
            evaluation_mask = mask * (1 - artificial_mask) # This gives 1 where originally observed and now masked, 0 otherwise

            if evaluation_mask.sum() > 0: # Only calculate if there are artificially masked values to evaluate
                 acc_masked, mae_masked, rmse_masked = calculate_metrics(logits, x, evaluation_mask)
                 val_accuracy_masked += acc_masked
                 val_mae_masked += mae_masked
                 val_rmse_masked += rmse_masked

            num_val_batches += 1

            # Update tqdm progress bar with current loss and metrics
            # val_loop.set_postfix(loss=loss.item(), rec_loss=reconstruction_loss.item(), kl_loss_w=kl_loss.item()*beta,
            #                      acc=acc, mae=mae, rmse=rmse,
            #                      acc_masked=acc_masked, mae_masked=mae_masked, rmse_masked=rmse_masked) # Removed tqdm postifx


    # Average validation loss and metrics - Divide by number of batches processed
    avg_val_loss = val_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_reconstruction_loss = val_reconstruction_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_kl_loss = val_kl_loss / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_accuracy = val_accuracy / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_mae = val_mae / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_rmse = val_rmse / num_val_batches if num_val_batches > 0 else 0.0
    # Average masked metrics (only if there were batches with artificial masking applied)
    # This calculation seems overly complex and potentially incorrect if num_val_batches_with_artificial_mask is used
    # Let's simplify by dividing by total number of validation batches if masked metrics were accumulated.
    # If no batches had artificial masking, the sum will be 0 and the average will be 0.
    avg_val_accuracy_masked = val_accuracy_masked / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_mae_masked = val_mae_masked / num_val_batches if num_val_batches > 0 else 0.0
    avg_val_rmse_masked = val_rmse_masked / num_val_batches if num_val_batches > 0 else 0.0


    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}, Rec Loss: {avg_train_reconstruction_loss:.4f}, KL Loss (weighted): {avg_train_kl_loss:.4f}, Beta: {beta:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Val Metrics (Observed): Acc: {avg_val_accuracy:.4f}, MAE: {avg_val_mae:.4f}, RMSE: {avg_val_rmse:.4f}")
    print(f"  Val Metrics (Masked): Acc: {avg_val_accuracy_masked:.4f}, MAE: {avg_val_mae_masked:.4f}, RMSE: {avg_val_rmse_masked:.4f}")


    # Early Stopping Check (only after warm-up epochs)
    if epoch >= BETA_WARMUP_EPOCHS:
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # Optionally save the best model
            # torch.save(model.state_dict(), 'best_cvae_model.pth')
            print("Validation loss improved. Saving model...")
        else:
            epochs_without_improvement += 1
            print(f"Validation loss did not improve for {epochs_without_improvement} epochs.")
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
    else:
        # During warm-up, just print improvement status relative to best_val_loss
        if avg_val_loss < best_val_loss:
             best_val_loss = avg_val_loss
             print("Validation loss improved during warm-up.")
        else:
             print("Validation loss did not improve during warm-up.")


print("Training finished.")

# After training, you can load the best model (if saved) and run final evaluation
# Example:
# try:
#     model.load_state_dict(torch.load('best_cvae_model.pth'))
#     print("Loaded best model for final evaluation.")
# except FileNotFoundError:
#     print("No best model saved, using the last trained model.")

# Final evaluation on the validation set (or a dedicated test set)
# Calculate final metrics using the best model
# ... (similar to the validation loop, but without training steps or early stopping)

Using device: cuda
Starting training...
  Epoch [1/100], Batch [1000], Loss: 0.7763, Rec Loss: 0.7763, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [2000], Loss: 0.6048, Rec Loss: 0.6048, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [3000], Loss: 0.5388, Rec Loss: 0.5388, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [4000], Loss: 0.4444, Rec Loss: 0.4444, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [5000], Loss: 0.3788, Rec Loss: 0.3788, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [6000], Loss: 0.3240, Rec Loss: 0.3240, KL Loss (weighted): 0.0000, Beta: 0.0000
  Epoch [1/100], Batch [7000], Loss: 0.2886, Rec Loss: 0.2886, KL Loss (weighted): 0.0000, Beta: 0.0000

Epoch [1/100] Summary:
  Train Loss: 0.5114, Rec Loss: 0.5114, KL Loss (weighted): 0.0000, Beta: 0.0000
  Val Loss: 0.2498
  Val Metrics (Observed): Acc: 0.9042, MAE: 0.1879, RMSE: 0.6925
  Val Metrics (Masked): Acc: 0.9040, MAE: 0

In [ ]:
import torch
import os

# 예: '/content/drive/MyDrive/Projects/CVAE_MI/cvae_ordinal_model.pt'
# 현재 날짜/시간 등을 포함하여 파일명을 동적으로 생성하는 것도 좋은 방법입니다.
model_save_path = '/content/drive/MyDrive/Projects/CVAE_MI/model_0628.pt' # 예시 파일명

# 저장할 디렉토리가 없으면 생성
save_dir = os.path.dirname(model_save_path)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"디렉토리 생성됨: {save_dir}")

# 모델의 state_dict 저장
try:
    torch.save(model.state_dict(), model_save_path)
    print(f"모델이 다음 경로에 저장되었습니다: {model_save_path}")
except Exception as e:
    print(f"모델 저장 중 오류가 발생했습니다: {e}")

# 데이터 로딩 및 전처리 (평가 자동화에 사용)

In [ ]:
import pandas as pd
import numpy as np
import torch

# Use the file path defined in the hyperparameters cell (QT_6WTgCfUN3)
# FILE_PATH = "/content/drive/MyDrive/Projects/CVAE_MI/big5_ans_only.csv" # This was for training data
# For evaluation, let's use the original full data file used in cell B0Lq7FjT5kTA
EVAL_FILE_PATH = '/content/drive/MyDrive/Projects/CVAE_MI/big5_100K.csv'


# Load the original data for evaluation
try:
    # Assuming the evaluation data file is the one without introduced NaNs (used to get ground truth)
    # Let's load it and remove rows with *any* 0/NaN to get a clean base for artificial masking
    df_raw_eval = pd.read_csv(EVAL_FILE_PATH)
    print(f"원본 평가 데이터 파일 '{EVAL_FILE_PATH}'을 성공적으로 불러왔습니다.")

    # Remove rows with any 0s (assuming 0 is not a valid Likert score and represents missing/invalid in this context)
    # This creates the 'ground truth' dataframe for evaluation *before* artificial masking
    rows_with_zero_or_nan = (df_raw_eval == 0).any(axis=1) | df_raw_eval.isna().any(axis=1)
    df_original_eval = df_raw_eval[~rows_with_zero_or_nan].copy()

    print(f"0 또는 NaN을 포함하는 행을 삭제한 후 평가 데이터 행 수: {len(df_original_eval)}")
    display(df_original_eval.describe())

except FileNotFoundError:
    print(f"오류: 평가 데이터 파일을 찾을 수 없습니다. 경로를 확인하세요: {EVAL_FILE_PATH}")
except Exception as e:
    print(f"평가 데이터를 읽는 중 오류가 발생했습니다: {e}")

# Get the list of columns to process (assuming all columns in df_original_eval are items)
columns_to_process = df_original_eval.columns.tolist()
NUM_ITEMS_EVAL = len(columns_to_process) # Update NUM_ITEMS if necessary for evaluation data

# Ensure the NUM_ITEMS used for model definition matches the data
# If your training data had a different number of items, you need to handle that.
# Assuming the model was trained on data with the same number of items as EVAL_FILE_PATH.
if 'NUM_ITEMS' not in globals() or NUM_ITEMS != NUM_ITEMS_EVAL:
    print(f"경고: 학습 시 사용된 NUM_ITEMS ({globals().get('NUM_ITEMS', 'N/A')})와 평가 데이터의 아이템 수 ({NUM_ITEMS_EVAL})가 다릅니다. 모델 정의와 일치하는지 확인하세요.")
    # Update NUM_ITEMS for consistency if needed, but be cautious if model was trained with different size
    # NUM_ITEMS = NUM_ITEMS_EVAL # Uncomment if you are sure this is intended


# --- 데이터 전처리 함수 (평가 자동화에 사용) ---
def preprocess_for_imputation(df_subset, columns):
    """
    Imputation 평가를 위해 DataFrame을 모델 입력 형식으로 전처리합니다.
    - df_subset: pandas DataFrame (인위적 결측이 포함된 데이터)
    - columns: 처리할 컬럼 리스트

    결측치는 0으로 변환하고, 마스크 및 조건 벡터 c를 생성하여 tensor 반환
    """
    df_proc = df_subset[columns].copy()

    # 원본 결측치 (NaN)는 0으로 채웁니다.
    # introduce_missing_values 함수가 이미 NaN을 생성했으므로, 여기서는 fillna(0)만 필요합니다.
    chunk_filled = df_proc.fillna(0).astype(int) # Ensure integer type for LongTensor

    # 마스크: NaN이 아닌 값은 1, NaN인 값은 0
    # introduce_missing_values에서 NaN으로 만든 위치가 mask에서 0이 됩니다.
    mask = (~df_proc.isna()).astype(float)

    # 각 샘플의 관측된 값에 대한 열 평균 계산
    # .values를 사용하여 인덱스 정렬 문제 방지
    observed_values = chunk_filled.values * mask.values # mask가 0이면 0이 됨
    sum_observed = np.nansum(observed_values, axis=1) # 0을 nan처럼 처리하지 않으므로 sum 사용 가능
    num_observed = np.sum(mask.values, axis=1)
    # 0으로 나누는 경우 방지
    per_sample_means = np.divide(sum_observed, num_observed, out=np.zeros_like(sum_observed), where=num_observed != 0)
    per_sample_means = per_sample_means[:, np.newaxis] # (n_samples, 1) 형태로 재구성

    # 조건 벡터 c 생성: 마스크와 열 평균 결합
    # shapes: mask.values (N, num_items), per_sample_means (N, 1)
    # C_DIM은 num_items (mask) + 1 (mean) 이어야 합니다.
    # 학습 시 정의된 C_DIM과 일치하는지 확인해야 합니다.
    if 'C_DIM' not in globals() or C_DIM != NUM_ITEMS_EVAL + 1:
         print(f"경고: 학습 시 사용된 C_DIM ({globals().get('C_DIM', 'N/A')})과 평가 데이터의 조건 벡터 차원 ({NUM_ITEMS_EVAL + 1})이 다릅니다. 모델 정의와 일치하는지 확인하세요.")
         # C_DIM = NUM_ITEMS_EVAL + 1 # Uncomment if you are sure this is intended


    c = np.concatenate([mask.values, per_sample_means], axis=1)


    # tensor 변환
    # chunk_filled는 모델 입력 x로 사용 (0~5 값 포함) -> LongTensor로 변환
    # mask는 마스크로 사용 (0 또는 1) -> FloatTensor 유지
    # c는 조건 벡터로 사용 -> FloatTensor 유지
    return torch.LongTensor(chunk_filled.values), torch.FloatTensor(mask.values), torch.FloatTensor(c)

# --- 인위적 결측 생성 함수 (평가 자동화에 사용) ---
# introduce_missing_values 함수는 이미 정의되어 있으므로 다시 정의하지 않습니다.
# cell_id: 20e18340에 정의된 함수를 사용합니다.

# 모델 로드 및 평가 자동화

In [ ]:
import torch
import pandas as pd
import numpy as np
# Ensure CVAE, OrdinalEmbedding, calculate_metrics, introduce_missing_values, preprocess_for_imputation are defined in previous cells

# 모델을 저장했던 경로와 동일해야 합니다.
# cell xdzukH359run에서 저장된 경로를 사용합니다.
model_save_path = '/content/drive/MyDrive/Projects/CVAE_MI/model_0628.pt'

# 학습 시 사용했던 하이퍼파라미터와 일치시켜 모델 인스턴스 생성
# QT_6WTgCfUN3 셀에서 정의된 값들을 사용합니다.
# NUM_ITEMS, EMBEDDING_DIM, C_DIM, LATENT_DIM
# Ensure these variables are available from previous cells

# Check if required hyperparameters are defined
required_hyperparameters = ['NUM_ITEMS', 'EMBEDDING_DIM', 'C_DIM', 'LATENT_DIM']
for hp in required_hyperparameters:
    if hp not in globals():
        raise NameError(f"필수 하이퍼파라미터 '{hp}'가 정의되지 않았습니다. QT_6WTgCfUN3 셀을 실행했는지 확인하세요.")

# Ensure NUM_ITEMS is consistent with the evaluation data loaded
# NUM_ITEMS_EVAL was calculated in the previous cell
if 'NUM_ITEMS_EVAL' not in globals():
     raise NameError("평가 데이터 아이템 수 (NUM_ITEMS_EVAL)가 정의되지 않았습니다. 평가 데이터 로딩 셀을 실행했는지 확인하세요.")

if NUM_ITEMS != NUM_ITEMS_EVAL:
    print(f"경고: 모델 정의에 사용된 NUM_ITEMS ({NUM_ITEMS})와 평가 데이터의 아이템 수 ({NUM_ITEMS_EVAL})가 다릅니다. 모델 로드가 예상대로 작동하지 않을 수 있습니다.")
    # Decide how to proceed: either stop, or assume model can handle different sizes (unlikely without code changes)
    # For now, proceed but with a warning. If the model's linear layers depend on NUM_ITEMS, this will fail.


# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"모델 로드 및 평가에 사용될 디바이스: {device}")

# 모델 인스턴스 생성 - 학습 시 사용했던 파라미터와 동일하게 전달
loaded_model = CVAE(num_items=NUM_ITEMS,
                    embedding_dim=EMBEDDING_DIM,
                    c_dim=C_DIM,
                    latent_dim=LATENT_DIM)

# 저장된 state_dict 로드
try:
    loaded_model.load_state_dict(torch.load(model_save_path, map_location=device))
    print(f"모델이 '{model_save_path}' 경로에서 성공적으로 로드되었습니다.")
except FileNotFoundError:
    print(f"오류: 모델 파일을 찾을 수 없습니다. 경로를 확인하세요: {model_save_path}")
    # Exit or handle the error appropriately if the model file is crucial
    raise # Re-raise the exception to stop execution if model loading fails
except Exception as e:
    print(f"모델 로드 중 오류가 발생했습니다: {e}")
    raise # Re-raise other exceptions


# 모델을 설정한 디바이스로 이동 및 평가 모드 설정
loaded_model.to(device)
loaded_model.eval()


# --- 결측치 복원 함수 (평가 자동화에 사용) ---
def impute_missing_values_with_cvae(model, data_tensor_encoded, mask_tensor, c_tensor, device):
    """
    학습된 CVAE 모델을 사용하여 결측치를 복원합니다.
    모델의 forward pass를 사용하여 logits을 얻고, 이를 바탕으로 예측값을 결정합니다.

    Args:
        model: 학습된 CVAE 모델 (eval mode)
        data_tensor_encoded: 결측치 0으로 인코딩된 데이터 tensor (N, num_items)
        mask_tensor: 마스크 tensor (N, num_items), 1 for observed, 0 for missing
        c_tensor: 조건 벡터 tensor (N, c_dim)
        device: cpu or cuda

    Returns:
        torch.Tensor: imputed tensor (N, num_items), int (1~5)
    """
    model.eval() # Ensure model is in evaluation mode
    data_tensor_encoded = data_tensor_encoded.to(device)
    # mask_tensor는 사용하지 않지만 to(device) 유지
    mask_tensor = mask_tensor.to(device)
    c_tensor = c_tensor.to(device)


    with torch.no_grad(): # Disable gradient calculation
        # 모델의 forward pass 실행. 인코더는 data_tensor_encoded (0-filled)와 c_tensor 사용
        # 디코더는 샘플링된 z와 c_tensor 사용 -> logits 반환
        logits, mu, logvar = model(data_tensor_encoded, c_tensor) # logits shape: (N, num_items, num_classes)

        # 예측된 클래스 (가장 확률 높은 리커트 점수)
        # num_classes는 모델의 Decoder 출력 차원에서 암시적으로 결정됩니다 (현재 5)
        predicted_categories = torch.argmax(logits, dim=2) # shape: (N, num_items), 0-indexed (0~4)

        # 0-indexed 카테고리를 1-indexed 리커트 점수 (1~5)로 변환
        imputed_vals = predicted_categories + 1 # shape (N, num_items), values 1-5


        # 최종 복원된 텐서 생성:
        # 원본 인코딩된 텐서 (관측값은 1-5, 결측치는 0)로 시작
        imputed_tensor_final = data_tensor_encoded.clone()

        # 원래 결측치였던 위치 (마스크가 0) 식별
        missing_mask = (mask_tensor == 0) # shape: (N, num_items) -> boolean

        # 최종 텐서에서 결측치 위치의 값을 복원된 값으로 대체
        imputed_tensor_final[missing_mask] = imputed_vals[missing_mask]


    return imputed_tensor_final.cpu() # Return tensor on CPU


# --- 평가 자동화 루프 ---
# df_original_eval은 이전 셀에서 로드되고 0/NaN이 제거된 깨끗한 데이터입니다.
# columns_to_process는 df_original_eval의 컬럼 리스트입니다.

result = []
# Define the list of columns to introduce missingness and evaluate - already done as columns_to_process

# Set the device - already done above
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Already defined

# Ensure the loaded_model is on the correct device - already done above
# loaded_model.to(device)
# loaded_model.eval() # Already set above


print("--- 평가 자동화 시작 ---")

# Loop through missingness types
for mis_type in ['random', 'negative_uni', 'positive_uni', 'bidirection']:
    # Loop through missingness rates
    for mis_rate in [0.10, 0.30, 0.50]:
        print(f"\n시나리오 처리 중: missingness_type='{mis_type}', missingness_rate={mis_rate:.2f}")

        # 각 시나리오마다 원본 데이터프레임의 깨끗한 복사본으로 시작
        df_scenario = df_original_eval.copy()

        # 각 컬럼에 대해 현재 mis_type과 mis_rate로 인위적 결측 생성
        # introduce_missing_values 함수는 컬럼별로 적용됩니다.
        for col in columns_to_process:
             df_scenario = introduce_missing_values(df_scenario, col, mis_type, mis_rate)

        # --- 복원 단계 ---
        # 복원을 위해 데이터 전처리 (결측치를 0으로 채우고 마스크 및 c 생성)
        data_tensor_encoded, mask_tensor, c_tensor = preprocess_for_imputation(df_scenario, columns_to_process)
        # Note: preprocess_for_imputation returns tensors on CPU, move to device inside imputation function


        # 학습된 모델을 사용하여 결측치 복원
        # num_categories는 모델의 출력 레이어에서 암시적으로 5로 가정됩니다.
        imputed_tensor = impute_missing_values_with_cvae(loaded_model, data_tensor_encoded, mask_tensor, c_tensor, device)


        # 복원된 텐서를 pandas DataFrame으로 변환
        df_imputed_scenario = pd.DataFrame(imputed_tensor.cpu().numpy(), columns=columns_to_process, index=df_scenario.index)


        # --- 평가 단계 ---
        # df_scenario (인위적 결측이 있는 데이터)에서 현재 NaN인 셀에 대한 boolean 마스크 생성
        current_missing_mask = df_scenario.isna()

        # df_original_eval에서 현재 결측인 위치의 실제 값 가져오기 (ground truth)
        # df_original_eval과 df_scenario는 인덱스가 동일하므로 .loc 사용 가능
        original_missing_values_series = df_original_eval[current_missing_mask].stack()
        original_missing_values = original_missing_values_series.values

        # df_imputed_scenario에서 df_scenario에서 결측이었던 위치의 복원된 값 가져오기
        imputed_values_for_missing_series = df_imputed_scenario[current_missing_mask].stack()
        imputed_values_for_missing = imputed_values_for_missing_series.values

        # 인위적으로 결측이 생성되고 복원된 값이 있는지 확인
        if len(original_missing_values) == 0 or len(imputed_values_for_missing) == 0:
            print(f"  시나리오 mis_type='{mis_type}', mis_rate={mis_rate:.2f}에서 평가할 결측치가 없습니다. 메트릭 계산을 건너뜁니다.")
            result.append({
                'mis_type': mis_type,
                'mis_rate': mis_rate,
                'accuracy': np.nan,
                'mae': np.nan,
                'rmse': np.nan
            })
            continue

        if len(original_missing_values) != len(imputed_values_for_missing):
            print(f"  오류: 시나리오 mis_type='{mis_type}', mis_rate={mis_rate:.2f}에서 원본 결측치 수({len(original_missing_values)})와 복원된 값 수({len(imputed_values_for_missing)})가 일치하지 않습니다. 메트릭 계산을 건너뜁니다.")
            result.append({
                'mis_type': mis_type,
                'mis_rate': mis_rate,
                'accuracy': np.nan,
                'mae': np.nan,
                'rmse': np.nan
            })
            continue

        # 계산을 위해 numpy 배열로 변환하고 정수형으로 변환
        original_missing_values = original_missing_values.astype(int)
        imputed_values_for_missing = imputed_values_for_missing.astype(int)

        # Imputation Accuracy 계산
        accuracy = np.mean(original_missing_values == imputed_values_for_missing)

        # Mean Absolute Error (MAE) 계산
        mae = np.mean(np.abs(original_missing_values - imputed_values_for_missing))

        # Root Mean Squared Error (RMSE) 계산
        rmse = np.sqrt(np.mean((original_missing_values - imputed_values_for_missing)**2))

        # 결과 리스트에 추가
        result.append({
            'mis_type': mis_type,
            'mis_rate': mis_rate,
            'accuracy': accuracy,
            'mae': mae,
            'rmse': rmse
        })

        print(f"  시나리오 완료: Acc: {accuracy:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")


print("\n--- 평가 자동화 완료 ---")

# 결과를 DataFrame으로 변환하여 출력
results_df = pd.DataFrame(result)

print("\n평가 결과 요약:")
display(results_df)